# Backtest Runner

Orchestrates running strategies against instruments using NautilusTrader's BacktestEngine.
Results are written to `backtest_catalog/` as feather files for the dashboard.

## Available Strategies
- **BBBStrategy** — Bollinger Band Breakout with optional MA trend filtering

## Available Instruments
- **XAUUSD** — Gold vs US Dollar (CFD, 1-min bars from IB)

In [ ]:
import os
from decimal import Decimal
from pathlib import Path
from dataclasses import dataclass

import pandas as pd
import pyarrow.ipc as ipc

from nautilus_trader.backtest.config import BacktestEngineConfig
from nautilus_trader.backtest.engine import BacktestEngine
from nautilus_trader.config import LoggingConfig
from nautilus_trader.model.currencies import USD
from nautilus_trader.model.data import Bar, BarType
from nautilus_trader.model.enums import AccountType, OmsType
from nautilus_trader.model.identifiers import InstrumentId, Symbol, TraderId, Venue
from nautilus_trader.model.instruments import CurrencyPair
from nautilus_trader.model.objects import Currency, Money, Price, Quantity
from nautilus_trader.persistence.config import StreamingConfig
from nautilus_trader.serialization.arrow.serializer import ArrowSerializer

from strategies import (
    ArrayKind,
    BandKind,
    BBBSignalVariant,
    BBBStrategy,
    BBBStrategyConfig,
    MATrendKind,
)

## Instrument Definitions

In [ ]:
SIM = Venue("SIM")

XAUUSD_SIM = CurrencyPair(
    instrument_id=InstrumentId(Symbol("XAU/USD"), SIM),
    raw_symbol=Symbol("XAU/USD"),
    base_currency=Currency.from_str("XAU"),
    quote_currency=Currency.from_str("USD"),
    price_precision=2,
    size_precision=0,
    price_increment=Price.from_str("0.01"),
    size_increment=Quantity.from_int(1),
    lot_size=None,
    max_quantity=None,
    min_quantity=Quantity.from_int(1),
    max_price=None,
    min_price=None,
    margin_init=Decimal("0.03"),
    margin_maint=Decimal("0.03"),
    maker_fee=Decimal("0.00002"),
    taker_fee=Decimal("0.00002"),
    ts_event=0,
    ts_init=0,
)

# Registry of available instruments
INSTRUMENTS = {
    "XAUUSD": XAUUSD_SIM,
}

## Load Bar Data from Catalog

Reads existing bar data from a previous backtest run in the catalog.
This avoids needing an external CSV — the catalog already contains XAUUSD bar data.

In [ ]:
CATALOG_PATH = Path(os.environ.get(
    "NAUTILUS_STORE_PATH",
    str(Path(__file__).resolve().parents[3] / "backtest_catalog"),
))

def find_bar_data(catalog_path: Path, instrument_prefix: str) -> tuple[Path, BarType] | None:
    """Find the first run containing bar data for the given instrument."""
    backtest_dir = catalog_path / "backtest"
    if not backtest_dir.exists():
        return None

    for run_dir in sorted(backtest_dir.iterdir()):
        bar_dir = run_dir / "bar"
        if not bar_dir.exists():
            continue
        for bar_type_dir in bar_dir.iterdir():
            if instrument_prefix in bar_type_dir.name:
                feather_files = list(bar_type_dir.glob("*.feather"))
                if feather_files:
                    bar_type = BarType.from_str(bar_type_dir.name)
                    return feather_files[0], bar_type
    return None

def load_bars_from_catalog(feather_path: Path, bar_type: BarType) -> list[Bar]:
    """Load Bar objects from a feather file using NautilusTrader's ArrowSerializer."""
    with open(feather_path, "rb") as f:
        reader = ipc.open_stream(f)
        table = reader.read_all()
    return ArrowSerializer.deserialize(Bar, table)

result = find_bar_data(CATALOG_PATH, "XAUUSD")
if result is None:
    raise FileNotFoundError(f"No XAUUSD bar data found in {CATALOG_PATH}")

feather_path, bar_type = result
bars = load_bars_from_catalog(feather_path, bar_type)
print(f"Loaded {len(bars)} bars of type {bar_type}")
print(f"From: {bars[0].ts_init} to {bars[-1].ts_init}")

## Configure and Run Backtest

Set up the strategy config and run the backtest engine.
Results are streamed to the catalog for the dashboard to pick up.

In [ ]:
instrument = INSTRUMENTS["XAUUSD"]

strategy_config = BBBStrategyConfig(
    instrument_id=instrument.id,
    bar_type=bar_type,
    trade_size=Decimal("1"),
    buy_array_kind=ArrayKind.CLOSE,
    buy_band_kind=BandKind.TOP,
    buy_period=20,
    buy_sd=1.0,
    sell_array_kind=ArrayKind.CLOSE,
    sell_band_kind=BandKind.TOP,
    sell_period=20,
    sell_sd=3.0,
    frequency_bars=10,
    signal_variant=BBBSignalVariant.BASELINE,
)

engine_config = BacktestEngineConfig(
    trader_id=TraderId("BACKTESTER-001"),
    logging=LoggingConfig(log_level="INFO"),
    streaming=StreamingConfig(
        catalog_path=str(CATALOG_PATH),
        replace_existing=True,
    ),
)

engine = BacktestEngine(config=engine_config)

engine.add_venue(
    venue=SIM,
    oms_type=OmsType.NETTING,
    account_type=AccountType.MARGIN,
    base_currency=USD,
    starting_balances=[Money(100_000, USD)],
)

engine.add_instrument(instrument)
engine.add_data(bars)
engine.add_strategy(strategy=BBBStrategy(config=strategy_config))

print("Running backtest...")
engine.run()
print("Backtest complete!")

## Results Summary

In [ ]:
with pd.option_context(
    "display.max_rows", 100,
    "display.max_columns", None,
    "display.width", 300,
):
    print("=== Account Report ===")
    print(engine.trader.generate_account_report(SIM))
    print("\n=== Positions Report ===")
    print(engine.trader.generate_positions_report())
    print("\n=== Order Fills Report ===")
    print(engine.trader.generate_order_fills_report())

In [ ]:
engine.reset()
engine.dispose()
print(f"Results written to {CATALOG_PATH}")
print("Run the dashboard (bun run dev) to view results.")